# ДЗ 5. Первый взгляд на orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw05` репозитория `dsa-homework` (`hw05.ipynb`): `git add hw05` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: Pandas с урока 5 и всё, что было раньше. `groupby`, `apply`, `.str`-методы (кроме готовой строки для `amount`) пока не используем. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([10, 20, 30], index=["a", "b", "c"])
print(s["b"], s.iloc[0])
# 2
a = pd.Series({"Алматы": 5, "Шымкент": 2})
b = pd.Series({"Шымкент": 1, "Астана": 4})
print(a + b)
# 3
df = pd.DataFrame({"x": [1, 2, 3], "y": [4, 5, 6]})
print(df[df["x"] > 1].shape)
# 4
print(type(df["x"]).__name__, type(df[["x"]]).__name__)
# 5
f = df[df["x"] > 1]
print(f.index.tolist(), f.iloc[0, 1])
```

Ваши ответы и объяснения:

1. 20 10. s["b"] берёт значение по названию индекса (метка "b" это 20), а s.iloc[0] берёт по позиции, первый элемент, то есть 10.
2. Алматы NaN, Астана NaN, Шымкент 3.0. Pandas складывает серии по названиям индекса, а не по порядку. Шымкент есть в обеих сериях (2 + 1 = 3). Алматы есть только в a, Астана только в b, второго слагаемого нет, поэтому получается NaN. Из-за NaN числа становятся дробными (3.0).
3. (2, 2). Условие x > 1 оставляет две строки (где x равно 2 и 3), а столбцов по-прежнему два: x и y.
4. Series DataFrame. С одинарными скобками df["x"] получаем один столбец (Series). С двойными df[["x"]] получаем таблицу из одного столбца (DataFrame).
5. [1, 2] 5. После фильтра остались строки с индексами 1 и 2, индекс не сбрасывается. iloc[0, 1] берёт первую строку и второй столбец, то есть значение y в строке с индексом 1, это 5.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("orders.csv")
# Готовая строка: суммы вида "15 990" записаны текстом. Убираем пробел и переводим в числа (подробно на уроке 6)
df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
print(df.shape)

(5463, 8)


In [ ]:
# Даты записаны текстом "2026-09-01 08:00", поэтому строки можно сравнивать
sep = df["order_date"] < "2026-10"     # сентябрь
octo = ~sep                            # октябрь
paid = df["status"] == "оплачен"
not_test = df["customer_id"] != "TEST"
SH = ["Шымкент", "шымкент", "Shymkent"]
print("сентябрь:", sep.sum(), "| октябрь:", octo.sum())

сентябрь: 2782 | октябрь: 2681


## [База] 1. Паспорт таблицы
Сколько строк и столбцов? Какие типы у столбцов? Сколько пропусков в `delivery_days`? Сколько разных значений в `city`, `category`, `status`, `customer_id`?
Напишите менеджеру три проблемы данных, которые видно уже сейчас, и чем каждая опасна для отчёта о выручке.

In [ ]:
# 1

In [ ]:
df

,order_id,order_date,customer_id,city,category,amount,delivery_days,status
0,101674,2026-09-01 08:00,C00681,Алматы,Одежда,8070.0,1.0,оплачен
1,101214,2026-09-01 08:00,C01210,Астана,Красота,3630.0,2.0,оплачен
2,102424,2026-09-01 08:00,C01408,Караганда,Дом,9120.0,3.0,оплачен
3,102333,2026-09-01 08:00,C01529,Алматы,Одежда,9350.0,2.0,оплачен
4,102564,2026-09-01 09:00,C00657,Астана,Дом,11010.0,1.0,оплачен
...,...,...,...,...,...,...,...,...
5458,104168,2026-10-31 21:00,C01057,Алматы,Одежда,11430.0,2.0,оплачен
5459,104648,2026-10-31 21:00,C00629,Караганда,Книги,3640.0,2.0,оплачен
5460,104615,2026-10-31 22:00,C00557,Астана,Электроника,22150.0,NaN,оплачен
5461,103460,2026-10-31 22:00,C02162,Алматы,Электроника,24670.0,2.0,отменён


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5463 entries, 0 to 5462
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       5463 non-null   int64  
 1   order_date     5463 non-null   object 
 2   customer_id    5463 non-null   object 
 3   city           5463 non-null   object 
 4   category       5463 non-null   object 
 5   amount         5463 non-null   float64
 6   delivery_days  5303 non-null   float64
 7   status         5463 non-null   object 
dtypes: float64(2), int64(1), object(5)
memory usage: 341.6+ KB


In [ ]:
df["delivery_days"].isna().sum()

np.int64(160)

In [ ]:
a = len(df["city"].unique())
b = len(df["category"].unique())
c = len(df["status"].unique())
d = len(df["customer_id"].unique())
print(f"Кол-во уникальныйх городов:{a}, категорий:{b}, статусов:{c}, айди клиентов:{d}")

Кол-во уникальныйх городов:14, категорий:5, статусов:2, айди клиентов:2003


#### Первая проблема которая очень важна для отчета о выручки это столбец "city", а именно его уникальные значения которые дублируются и отличается только написанием. Вторая проблема это пустые знаения в столбце "delivery_days", из-за которых может исказиться аналитика по данному показателю. Данную проблему можно решить 2 спосабами: первое это удалить строки с пустыми значениями, второе заменить пустые значения средними показателями.К 3 проблеме можно отнести тип данных в "order_date", который явлется object  а не datetimе, что может услужнить дальнейших анализ по данному показателю в  техническом плане.



## [База] 2. Типичный заказ
Для оплаченных заказов без TEST: среднее, медиана, 25-й и 75-й перцентили суммы.
Какой чек «типичный»? Что честнее показать в отчёте, среднее или медиану, и почему?

In [ ]:
# 2


In [ ]:
print(F"Среднее:{df[(df["customer_id"] != "TEST") & (df["status"] == "оплачен")]["amount"].mean()}")

Среднее:14672.391348704346


In [ ]:
print(F"Медиана:{df[(df["customer_id"] != "TEST") & (df["status"] == "оплачен")]["amount"].median()}")

Медиана:10300.0


In [ ]:
print(F"25-й перцентиль:{np.percentile(df[(df["customer_id"] != "TEST") & (df["status"] == "оплачен")]["amount"],25)}")

25-й перцентиль:5880.0


In [ ]:
print(F"75-й перцентиль:{np.percentile(df[(df["customer_id"] != "TEST") & (df["status"] == "оплачен")]["amount"],75)}")

75-й перцентиль:18690.0


#### В отчете важнее будет показать медиану так как среднее довольно чувствительна к выбросам из за которых значение может не совсем корректно показывать картину всей действительности. Допустим есть 10 чеков, 9 из которых показывают значения чуть больше 1.000 тг, и 1 чек который показывает значение в 100.000 тг. и  среднее автоматом становиться около 10.900, но явлется ли он достоверным если все чеки кроме одной меньше 10.900. Медиана же в свою очередь делит уже упорядоченные значения пополам, 50% чеков меньше неё и 50% больше. Если чеков как у меня четное он берет среднее между занчениями 5 и 6, если 11, берет 6 значение.

## [База] 3. Отмены по городам
Доля отменённых заказов: по всем заказам, в Шымкенте, в Алматы (все написания каждого города через `isin`).
Какой город выделяется и во сколько раз?

In [ ]:
# 3

In [ ]:
df["city"].unique()

array(['Алматы', 'Астана', 'Караганда', 'Шымкент', 'алматы', 'Атырау',
       'шымкент', 'Shymkent', 'астана', 'Актобе', 'Astana ', 'АЛМАТЫ',
       ' Алматы', 'Almaty'], dtype=object)

In [ ]:
ALM = ['Алматы', 'алматы','АЛМАТЫ',' Алматы' ,'Almaty']
print(F"Доля отменных заказов в Алмате:{df[df["status"] == "отменён"]["city"].isin(ALM).sum()/df["city"].isin(ALM).sum()}")

Доля отменных заказов в Алмате:0.07861060329067641


In [ ]:
print(F"Доля отменных заказов в Шымкент:{df[df["status"] == "отменён"]["city"].isin(SH).sum()/df["city"].isin(SH).sum()}")

Доля отменных заказов в Шымкент:0.20398009950248755


In [ ]:
AlM_share = 0.07861060329067641
SH_share = 0.20398009950248755
total_share = (df["status"] == "отменён").mean()
print(f"Шымкент выделяется: в {SH_share / AlM_share} раза выше Алматы")
print(f"Шымкент к общей доле: в {SH_share / total_share} раза")
print(f"Алматы к общей доле:  в {AlM_share / total_share} раза")

Шымкент выделяется: в 2.5948166146014113 раза выше Алматы
Шымкент к общей доле: в 1.9934584679464928 раза
Алматы к общей доле:  в 0.7682463788496695 раза


#### Доля отменённых заказов по всем данным составляет около 10,2%. В Алматы она ниже среднего (7,9%), в Шымкенте заметно выше (20,4%). Шымкент выделяется: отмены там случаются почти в 2 раза чаще, чем в среднем по всем заказам, и в 2,6 раза чаще, чем в Алматы.

## [Стандарт] 4. Октябрь против сентября по двум городам
Выручка оплаченных заказов без TEST за сентябрь и октябрь в Шымкенте и в Алматы, изменение в %.
Дубли пока не убираем. Где падение сильнее и во сколько раз? Что сказать менеджеру?

In [ ]:
# 4

In [ ]:
df[(df["status"] == "оплачен") & (df["city"].isin(ALM)) & (df["customer_id"] != "TEST") & (df["order_date"] < "2026-10")]["amount"].sum()

np.float64(15523940.0)

In [ ]:
df[(df["status"] == "оплачен") & (df["city"].isin(ALM)) & (df["customer_id"] != "TEST") & (octo == True)]["amount"].sum()

np.float64(14643020.0)

In [ ]:
df[(df["status"] == "оплачен") & (df["city"].isin(SH)) & (df["customer_id"] != "TEST") & (df["order_date"] < "2026-10")]["amount"].sum()

np.float64(4949800.0)

In [ ]:
df[(df["status"] == "оплачен") & (df["city"].isin(SH)) & (df["customer_id"] != "TEST") & (octo == True)]["amount"].sum()

np.float64(4044010.0)

#### В Шымкенте падение сильнее примерно в 3,2 раза. Менеджеру: основное проблемное место Шымкент, причину стоит искать там

## [Стандарт] 5. Исправьте код от AI
AI написал код «отменённые заказы Шымкента». В нём **две** ошибки: одна роняет код, вторая тихо даёт неверное число. Найдите обе и исправьте.
```python
cancel_sh = df[df["city"] == "Шымкент" & df["status"] == "отменён"]
print(len(cancel_sh))
```

In [ ]:
cancel_sh = df[(df["city"].isin(SH)) & (df["status"] == "отменён")]
print(len(cancel_sh))

164


#### Как и было сказано в задаче, в коде 2 ошибки, первая ошибка техническая которая роняет код это ошибка со скобками когда мы ставим 2 условия внутри квадратных скобок, каждое условие берём в скобки (df["city"] == "Шымкент") & (df["status"] == "отменён"). 

#### Вторая ошибка которая дает неверное число это условия по городу Шымкент, мы указали один текст Шымкент, а как мы помним в нашей таблице город Шымкент указан в 3 вариациях, поэтому данный код выдает неверное число, меньшее от действительности.


## [Стандарт] 6. Самые частые клиенты
Кто сделал больше всего заказов (`value_counts` по `customer_id`)? Покажите все заказы этого клиента (`loc` с маской, столбцы дата, город, сумма, статус).
Есть ли среди его заказов повторяющиеся `order_id`? Что это значит для выручки, которую мы считали весь урок?

In [ ]:
# 6

In [ ]:
custom_max = df["customer_id"].value_counts().idxmax()

In [ ]:
df.loc[df["customer_id"] == custom_max,["customer_id","order_id", "order_date","city","amount","status"]]

,customer_id,order_id,order_date,city,amount,status
57,C01818,101125,2026-09-01 15:00,Шымкент,7350.0,оплачен
779,C01818,100096,2026-09-09 18:00,Шымкент,45270.0,оплачен
1371,C01818,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
1375,C01818,101504,2026-09-15 21:00,Алматы,8890.0,оплачен
2195,C01818,101860,2026-09-24 18:00,Астана,1490.0,оплачен
2617,C01818,101462,2026-09-29 09:00,Астана,22170.0,оплачен
2776,C01818,101495,2026-09-30 21:00,Алматы,20180.0,оплачен
3417,C01818,103416,2026-10-08 14:00,Караганда,4330.0,оплачен
4279,C01818,102644,2026-10-18 08:00,Астана,13600.0,оплачен
4610,C01818,104001,2026-10-21 19:00,Алматы,21990.0,оплачен


#### Мы видим с прошлого кода то что имеется дубликат, так как ордер айди повторяется и все остальные данные идентичные. Это значит что наша выручку которую мы решали до этого, становиться неверной.
#### Больше всех заказов у клиента C01818 (11 строк). Заказ 101504 в нём встречается дважды с одинаковыми данными (8 890 тг), это дубль. Значит, выручка, которую мы считали, завышена из-за повторов, и их надо убирать до расчётов.

## [Челлендж] 7. Выбросы по правилу IQR
Для оплаченных заказов без TEST: `IQR = Q3 − Q1`, граница = `Q3 + 1.5 * IQR`. Сколько заказов выше границы, какая это доля и какая у них категория чаще всего?
Сравните с ДЗ 4: по z-оценке выбросов было 87. Почему правила дают разное число?

In [ ]:
# 7

In [2]:
df_IQR = df[(df["status"] == "оплачен") & (df["customer_id"] != "TEST")]

NameError: name 'df' is not defined

In [3]:
IQR = df_IQR["amount"].quantile(0.75) - df_IQR["amount"].quantile(0.25)
IQR

NameError: name 'df_IQR' is not defined

In [4]:
IQR_frame = df_IQR["amount"].quantile(0.75) + 1.5 * IQR
IQR_frame

NameError: name 'df_IQR' is not defined

In [1]:
len(df_IQR[df_IQR["amount"] > IQR_frame])

NameError: name 'df_IQR' is not defined

In [ ]:
len(df_IQR[df_IQR["amount"] > IQR_frame]) / len(df_IQR)

0.06529279738828811

In [ ]:
df_IQR[df_IQR["amount"] > IQR_frame].value_counts("category")

,count
category,
Электроника,308
Одежда,10
Дом,2


#### IQR равен 12 810, верхняя граница 37 905 тг. Выше границы 320 заказов, это около 6,5% оплаченных заказов. Почти все из них (308 из 320) в категории Электроника.

#### В ДЗ 4 выбросов было 87, а здесь 320, потому что правила разные. В z-оценке считается среднее и отклонение, а на них сильно влияют дорогие заказы, поэтому порог получается высокий и под него попадает мало заказов. В IQR считаются квартили, дорогие заказы на них почти не влияют, поэтому граница ниже (37 905 тг) и выбросов находится больше. Ещё здесь дубли не убраны, это тоже немного прибавляет.

## [Челлендж] 8. Доля отмен для каждого написания города одной строкой
Поделите две Series: число отменённых заказов по `city` и число всех заказов по `city` (обе через `value_counts`). Отсортируйте.
Pandas делит по ярлыкам: проверьте это. Доля отмен у «Шымкент», «шымкент» и «Shymkent» сильно разная. Сколько заказов у каждого написания?
Можно ли судить о городе по одному написанию? Что нужно сделать с городами до анализа (это тема урока 6)?

In [ ]:
# 8